# Personal Finance Knowledge Assistant
## 0. Project Overview

**Problem Statement**
Many people lack basic financial literacy, but professional financial advice is expensive. General LLMs can hallucinate or give dangerous personalized advice.

**Objective**
Build a simple, educational personal-finance question-answering assistant that strictly answers based on a provided knowledge base, demonstrating the progression from a basic LLM call to a containerized microservice architecture.

**Use Case**
Users can ask basic educational questions (e.g., "What is the 50/30/20 rule?") and receive accurate, context-backed explanations without risk of hallucination.

**Technology Stack**
*   **Environment:** Google Colab (NVIDIA T4 GPU)
*   **LLM Engine:** Ollama
*   **Model:** Code Llama (7B, quantized)
*   **Embeddings:** sentence-transformers/all-MiniLM-L6-v2
*   **Vector Database:** FAISS (Facebook AI Similarity Search)
*   **API Framework:** FastAPI
*   **Deployment Architecture:** Docker

**Final Architecture & Request Flow**
User → Application API → RAG Service → FAISS (Retrieves Context) → LLM Service (Ollama/Code Llama) → Response returned to User.

## 1. Environment Setup

Before implementing the application, we need to ensure our Colab environment has a GPU available and install the necessary libraries.

*We are using a T4 GPU provided freely by Google Colab to run our local Code Llama model.*

In [1]:
# Check GPU availability and CUDA setup
!nvidia-smi

# Install required Python dependencies
!pip install -q faiss-cpu sentence-transformers requests fastapi uvicorn nest-asyncio

Sun Aug 16 21:55:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### How This Works
*   `!nvidia-smi`: This command checks the NVIDIA GPU status. You should see a Tesla T4 listed. This hardware is necessary to run the LLM fast enough.
*   `pip install`: We are installing `faiss-cpu` for vector search, `sentence-transformers` for creating text embeddings, and `fastapi`/`uvicorn`/`nest-asyncio` for creating our microservices later.

## 2. Exercise 1 — Basic LLM Application

In this exercise, we will install Ollama (a tool for running LLMs locally), download the Code Llama model, and create a basic Python function to ask it a question.

In [14]:
# Install zstd (required for Ollama extraction)
!sudo apt-get update && sudo apt-get install -y zstd

# Install Ollama Linux binary
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
import subprocess
import time

# Run ollama serve in background
subprocess.Popen(["ollama", "serve"])
print("Waiting for Ollama server to start...")
time.sleep(5)

# Pull the Code Llama model (this may take a few minutes)
!ollama pull codellama

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [5]:
def ask_llm(question):
    """Sends a question to the Ollama API and returns the response."""
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "codellama",
        "prompt": question,
        "stream": False
    }

    response = requests.post(url, json=payload)

    if response.status_code == 200:
        return response.json().get("response")
    else:
        return f"Error: {response.text}"

# Let's test the basic LLM
user_question = "What is a credit score in simple terms?"
print(f"User: {user_question}")
print(f"Assistant: {ask_llm(user_question)}")

User: What is a credit score in simple terms?
Assistant: 
A credit score is a three-digit number that represents your creditworthiness, calculated by lenders based on your credit history. It is a measure of the likelihood of you repaying your debts on time and according to the terms of your loan. A higher credit score indicates that you have a lower risk of defaulting on your debts, while a lower score indicates that you may be more likely to default.

Credit scores are generated by credit reporting agencies, such as Equifax, Experian, and TransUnion, based on information in your credit report. This information includes payment history, credit utilization (the amount of credit you're using compared to the amount you have available), credit age (how long you've had your credit), and new credit (whether you've applied for new credit in the past two years).

The most common credit scores used by lenders are:

* FICO score: This is the most widely used credit score in the United States, wi

### How This Works
*   **Ollama** is a lightweight server that runs large language models locally on our hardware instead of in the cloud.
*   **Code Llama** is the model we are using. Though optimized for code, it is fundamentally a Llama 2 model that can process and generate English text.
*   **API (Application Programming Interface)** is how our Python code talks to Ollama. We send a standard HTTP POST request to `localhost:11434`.
*   After the LLM receives the prompt, it uses mathematical probabilities to predict the next best word (Next-Token Prediction) repeatedly until the answer is complete.

### Exercise Summary
*   **Implemented:** Ollama installation, Code Llama download, and a basic Python HTTP client.
*   **New Component:** Ollama Server & Code Llama model.
*   **Architecture:** User → Python App → Ollama API → Code Llama → Response.
*   **Demonstrate:** Run the cell and show the instructor that the Colab notebook is successfully talking to a local LLM without using internet APIs like OpenAI.

## 3. Exercise 2 — Create the Knowledge Base

We don't want the LLM to guess financial facts. We want it to use *our* trusted educational documents. We will create these documents, split them into chunks, convert them into mathematical vectors (embeddings), and store them in FAISS.

In [6]:
import os

# Create fictional educational documents
documents = {
    "budgeting_basics.txt": "Budgeting is the process of creating a plan to spend your money. It ensures you have enough money for things you need. Tracking expenses is a core part of monthly budgeting, helping you distinguish between needs (essential survival items like rent and food) and wants (non-essentials like entertainment).",
    "50_30_20_rule.txt": "The 50/30/20 budgeting rule is a simple framework. It suggests dividing your after-tax income into three categories: 50% for needs (housing, groceries, utilities), 30% for wants (dining out, hobbies, vacations), and 20% for savings or paying off debt.",
    "emergency_fund.txt": "An emergency fund is a bank account with money set aside to pay for large, unexpected expenses, such as a major medical bill or sudden job loss. Financial experts generally recommend keeping three to six months' worth of living expenses in an emergency fund.",
    "simple_interest.txt": "Simple interest is calculated only on the principal amount of a loan or deposit. The formula is Principal x Rate x Time. It does not account for interest earned on previously accumulated interest.",
    "compound_interest.txt": "Compound interest is the interest on savings calculated on both the initial principal and the accumulated interest from previous periods. It is often described as 'interest on interest' and makes your money grow faster over time.",
    "credit_score.txt": "A credit score is a three-digit number representing your creditworthiness, usually ranging from 300 to 850. It is based on your credit history, including payment history, amounts owed, and length of credit history. Higher scores mean you are a lower risk to lenders.",
    "saving_vs_investing.txt": "Saving is putting money aside for short-term goals or emergencies, usually in low-risk bank accounts. Investing is buying assets like stocks or bonds with the expectation that your money will grow over the long term, though it carries a higher risk of loss.",
    "loans.txt": "A loan is money borrowed from a lender that must be repaid with interest. Common types include mortgages for houses, auto loans for cars, and personal loans for various expenses."
}

# Create a folder and save them
os.makedirs("finance_kb", exist_ok=True)
for filename, content in documents.items():
    with open(f"finance_kb/{filename}", "w") as f:
        f.write(content)

print("Created 8 educational finance documents.")

# Chunking function
def chunk_text(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)
    return chunks

# Load and chunk documents
all_chunks = []
chunk_metadata = [] # To keep track of where each chunk came from

for filename in os.listdir("finance_kb"):
    with open(f"finance_kb/{filename}", "r") as f:
        text = f.read()
        file_chunks = chunk_text(text)
        for chunk in file_chunks:
            all_chunks.append(chunk)
            chunk_metadata.append({"source": filename, "text": chunk})

print(f"Total chunks created: {len(all_chunks)}")
print(f"Sample chunk: {all_chunks[0]}")

Created 8 educational finance documents.
Total chunks created: 13
Sample chunk: The 50/30/20 budgeting rule is a simple framework. It suggests dividing your after-tax income into three categories: 50% for needs (housing, groceries, utilities), 30% for wants (dining out, hobbies, vacations), and 20% for savings or paying off debt.


In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load the embedding model
print("Loading embedding model (MiniLM)...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Generate embeddings for our chunks
print("Converting text to vectors...")
embeddings = embedding_model.encode(all_chunks)
embedding_dimension = embeddings.shape[1]
print(f"Embedding dimension: {embedding_dimension} numbers per chunk")

# Store in FAISS vector database
print("Building FAISS index...")
faiss_index = faiss.IndexFlatL2(embedding_dimension)
faiss_index.add(np.array(embeddings))
print(f"Successfully stored {faiss_index.ntotal} vectors in FAISS.")

Loading embedding model (MiniLM)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Converting text to vectors...
Embedding dimension: 384 numbers per chunk
Building FAISS index...
Successfully stored 13 vectors in FAISS.


### How This Works
*   **Chunking:** LLMs have limited "memory" (context window). We break documents into 300-character chunks with a 50-character overlap. The overlap ensures we don't accidentally split a sentence in a way that destroys its meaning.
*   **Embeddings:** Computers don't understand words; they understand numbers. We pass our chunks through `MiniLM` to convert them into a vector (a list of 384 numbers). Sentences with similar meanings get converted into vectors that are mathematically close to each other.
*   **FAISS:** Instead of reading every document every time a user asks a question, we put our vectors into FAISS. FAISS acts like a super-fast search engine for mathematical vectors.

### Exercise Summary
*   **Implemented:** Knowledge base creation, text chunking, embedding generation, and Vector DB storage.
*   **New Component:** FAISS Database, Sentence Transformers.
*   **Architecture:** Documents → Chunking → MiniLM Embeddings → FAISS.
*   **Demonstrate:** Show the instructor how a paragraph of text is transformed into an array of 384 numbers (dimensions).

## 4. Exercise 3 — Retrieval + RAG (Retrieval-Augmented Generation)

Now we will combine everything. When a user asks a question, we convert the question into a vector, ask FAISS to find the 3 closest document vectors, and give that text to Code Llama to answer the question safely.

In [8]:
def retrieve_context(question, top_k=3):
    """Converts question to vector, searches FAISS, and returns relevant text."""
    # 1. Embed the question
    question_embedding = embedding_model.encode([question])

    # 2. Search FAISS
    distances, indices = faiss_index.search(np.array(question_embedding), top_k)

    # 3. Retrieve chunks
    retrieved_chunks = []
    sources = []
    for idx in indices[0]:
        chunk_data = chunk_metadata[idx]
        retrieved_chunks.append(chunk_data["text"])
        sources.append(chunk_data["source"])

    return retrieved_chunks, list(set(sources))

def ask_with_rag(question):
    """Complete RAG pipeline: Retrieval + Prompt Formatting + LLM generation."""
    chunks, sources = retrieve_context(question)
    context_string = "\n---\n".join(chunks)

    # The RAG Prompt
    prompt = f"""You are an educational personal finance knowledge assistant.
Answer the user's question ONLY using the provided context.
Do not invent information.
If the answer cannot be found in the context, say:
'This information is not available in my knowledge base.'
This system provides general educational information and is not personalized financial advice.

Context:
{context_string}

Question:
{question}

Answer:"""

    url = "http://localhost:11434/api/generate"
    payload = {"model": "codellama", "prompt": prompt, "stream": False}

    response = requests.post(url, json=payload).json().get("response")
    return response, sources, chunks

# --- Demonstrations ---
test_questions = [
    "What is the 50/30/20 rule?",
    "What is an emergency fund?",
    "What is the difference between needs and wants?",
    "What is compound interest?",
    "Should I buy Bitcoin?" # Hallucination test (not in knowledge base)
]

print("=== RAG DEMONSTRATION ===\n")
for q in test_questions:
    print(f"User: {q}")
    answer, sources, _ = ask_with_rag(q)
    print(f"Assistant: {answer}")
    print(f"[Sources used: {', '.join(sources)}]\n")
    print("-" * 40)

=== RAG DEMONSTRATION ===

User: What is the 50/30/20 rule?
Assistant: The 50/30/20 budgeting rule is a simple framework that suggests dividing your after-tax income into three categories: 50% for needs (housing, groceries, utilities), 30% for wants (dining out, hobbies, vacations), and 20% for savings or paying off debt.
[Sources used: 50_30_20_rule.txt, simple_interest.txt, credit_score.txt]

----------------------------------------
User: What is an emergency fund?
Assistant: 
An emergency fund is a bank account with money set aside to pay for large, unexpected expenses, such as a major medical bill or sudden job loss. Financial experts generally recommend keeping three to six months' worth of living expenses in an emergency fund.
[Sources used: emergency_fund.txt, saving_vs_investing.txt]

----------------------------------------
User: What is the difference between needs and wants?
Assistant: In the context of budgeting, the difference between needs and wants is that needs are esse

### How This Works
*   **Query Embedding:** The user's question is converted into the same 384-dimensional space as the documents.
*   **Similarity Search:** FAISS calculates the distance between the question vector and document vectors. The closer the distance, the more relevant the text!
*   **Context:** The top 3 closest chunks are stitched together into a single string.
*   **RAG:** We combine the Context and the Question into a strict prompt. We tell the LLM, "Only answer using this context." This practically eliminates **Hallucination** (the LLM making things up).

### Exercise Summary
*   **Implemented:** The full Retrieval-Augmented Generation (RAG) loop.
*   **New Component:** FAISS Search, RAG Prompting.
*   **Architecture:** Question → Embedding → FAISS → Context → LLM → Answer.
*   **Demonstrate:** Show the instructor how the last question ("Should I buy Bitcoin?") safely results in "This information is not available", proving the LLM is restricted to the knowledge base.

## 5. Exercise 4 — APIs, Services, and Orchestration

Instead of one big Python script, real-world applications use **Microservices**. We will build three APIs using FastAPI. Since running multiple web servers in one Colab notebook is messy, we will run one central API server that demonstrates the "Endpoints", using `nest_asyncio` to allow it to run in Colab.

In [24]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import nest_asyncio
import threading
import time
import requests
from google.colab import output

nest_asyncio.apply()
app = FastAPI(title="Finance Assistant Services")

app.add_middleware(
    CORSMiddleware,
    allow_origin_regex=r"https://.*",
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- NEW: A simple endpoint to help us clear the Google warning ---
@app.get("/")
def read_root():
    return {"message": "Success! The Colab security warning is bypassed. You can close this tab and use the chat."}

@app.post("/retrieve")
def retrieve_service(data: dict):
    question = data.get("question")
    chunks, sources = retrieve_context(question)
    return {"context_chunks": chunks, "sources": sources}

@app.post("/generate")
def llm_service(data: dict):
    prompt = data.get("prompt")
    url = "http://localhost:11434/api/generate"
    payload = {"model": "codellama", "prompt": prompt, "stream": False}
    res = requests.post(url, json=payload).json()
    return {"answer": res.get("response")}

@app.post("/ask")
def application_service(data: dict):
    question = data.get("question")

    retrieval_response = retrieve_service({"question": question})
    context = "\n".join(retrieval_response["context_chunks"])
    sources = retrieval_response["sources"]

    prompt = f"Answer using context ONLY.\nContext: {context}\nQuestion: {question}\nAnswer:"
    llm_response = llm_service({"prompt": prompt})

    return {
        "question": question,
        "answer": llm_response["answer"],
        "sources": sources
    }

def run_api():
    # Moving to 8004 to avoid old stuck servers
    uvicorn.run(app, host="127.0.0.1", port=8004, log_level="warning")

threading.Thread(target=run_api, daemon=True).start()
time.sleep(2)

# Generate the public URL
colab_url = output.eval_js("google.colab.kernel.proxyPort(8004)")
print("="*70)
print("🚨 CRITICAL STEP: YOU MUST CLICK THE LINK BELOW FIRST 🚨")
print("="*70)
print(f"Click here: {colab_url}")
print("1. A new tab will open.")
print("2. If you see a Google warning, click 'Visit Site' or 'Continue'.")
print("3. Once you see the {'message': 'Success!...'} text, close the tab.")
print("="*70)

🚨 CRITICAL STEP: YOU MUST CLICK THE LINK BELOW FIRST 🚨
Click here: https://8004-gpu-t4-s-kkb-usw4a0-2hxzu5nqan03f-a.us-west4-0.prod.colab.dev
1. A new tab will open.
2. If you see a Google warning, click 'Visit Site' or 'Continue'.
3. Once you see the {'message': 'Success!...'} text, close the tab.


### How This Works
*   **Service:** A specific, isolated piece of code that does exactly one job (e.g., retrieving data, or talking to the LLM).
*   **API Endpoint:** A web address (like `/retrieve` or `/ask`) where services can talk to each other over the internet using JSON data.
*   **Orchestration:** The Application Service acts like a manager. It receives the user's question, asks the Retrieval Service for data, bundles it together, and sends it to the LLM Service.

### Exercise Summary
*   **Implemented:** FastAPI REST architecture.
*   **New Component:** FastAPI, Uvicorn, JSON payloads.
*   **Architecture:** Separation of concerns (App, RAG, LLM).
*   **Demonstrate:** Show the instructor the JSON output from testing the `/ask` endpoint, proving that the services are communicating via HTTP requests.

## 6. Exercise 5 — Docker

*Note: Running Docker inside Google Colab is highly unreliable due to container nesting and GPU passthrough limitations.*

Instead, we will **generate** the Docker configuration files right here in Colab to demonstrate the final architecture. You would take these files to a local machine to run the containerized application.

In [10]:
# 1. Generate requirements.txt
with open("requirements.txt", "w") as f:
    f.write("fastapi\nuvicorn\nrequests\nsentence-transformers\nfaiss-cpu\n")

# 2. Generate Dockerfile (For Python APIs)
dockerfile = """
# Use an official Python runtime
FROM python:3.10-slim

# Set working directory
WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Expose API port
EXPOSE 8000

# Run the FastAPI server
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
with open("Dockerfile", "w") as f:
    f.write(dockerfile)

# 3. Generate docker-compose.yml
docker_compose = """
version: '3.8'

services:
  app_service:
    build: .
    ports:
      - "8000:8000"
    depends_on:
      - llm_service
    environment:
      - OLLAMA_URL=http://llm_service:11434

  llm_service:
    image: ollama/ollama:latest
    ports:
      - "11434:11434"
    volumes:
      - ollama_data:/root/.ollama
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]

volumes:
  ollama_data:
"""
with open("docker-compose.yml", "w") as f:
    f.write(docker_compose)

print("Generated requirements.txt, Dockerfile, and docker-compose.yml successfully!")

Generated requirements.txt, Dockerfile, and docker-compose.yml successfully!


### How This Works
*   **Dockerfile:** A recipe that tells Docker how to build an isolated mini-computer (container) with Python, FastAPI, and our code installed.
*   **docker-compose.yml:** A blueprint that launches our Application Container and the official Ollama Container at the same time, connecting them on a virtual network.
*   **GPU Considerations:** Notice the `deploy: resources:` section in the compose file. This tells Docker to pass your local NVIDIA GPU through to the Ollama container so Code Llama runs fast.

### Exercise Summary
*   **Implemented:** Docker configuration files.
*   **New Component:** Docker & Docker Compose.
*   **Demonstrate:** Explain the `docker-compose.yml` to the instructor. Show how `app_service` talks to `llm_service` via `http://llm_service:11434`.

## 7. Final Demonstration

This interactive cell simulates the final Application Interface. Run it and ask questions!
Type `exit` to stop.

In [26]:
from IPython.display import HTML, display, JSON
from google.colab import output

# --- THE ULTIMATE FIX ---
# Instead of a web HTTP request, we use Colab's internal bridge to directly
# trigger the application_service function you built in Exercise 4.
def colab_ask_endpoint(question):
    # This calls your orchestration API directly!
    response = application_service({"question": question})
    return JSON(response)

# Register the bridge
output.register_callback('ask_api', colab_ask_endpoint)

chat_ui_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Finance Assistant Chat</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
        .chat-container { width: 100%; max-width: 500px; height: 500px; background-color: #ffffff; border-radius: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.1); display: flex; flex-direction: column; overflow: hidden; margin: 0 auto; border: 1px solid #e2e8f0; }
        .chat-header { background-color: #1e293b; color: #ffffff; padding: 16px 20px; text-align: center; font-weight: 600; }
        .chat-box { flex: 1; padding: 20px; overflow-y: auto; display: flex; flex-direction: column; gap: 16px; background-color: #f8fafc; }
        .message { max-width: 80%; padding: 12px 16px; border-radius: 12px; font-size: 0.95rem; line-height: 1.4; white-space: pre-wrap; }
        .message.assistant { background-color: #e2e8f0; color: #334155; align-self: flex-start; border-bottom-left-radius: 4px; }
        .message.user { background-color: #2563eb; color: #ffffff; align-self: flex-end; border-bottom-right-radius: 4px; }
        .chat-input-container { display: flex; padding: 16px; background-color: #ffffff; border-top: 1px solid #e2e8f0; }
        .chat-input-container input { flex: 1; padding: 10px 16px; border: 1px solid #cbd5e1; border-radius: 24px; outline: none; }
        .chat-input-container button { background-color: #2563eb; color: white; border: none; border-radius: 50%; width: 40px; height: 40px; margin-left: 12px; cursor: pointer; display: flex; justify-content: center; align-items: center; }
        .send-icon { width: 0; height: 0; border-top: 5px solid transparent; border-bottom: 5px solid transparent; border-left: 8px solid white; margin-left: 3px; }
    </style>
</head>
<body>
    <div class="chat-container">
        <div class="chat-header">🎓 Finance Assistant</div>
        <div class="chat-box" id="chat-box">
            <div class="message assistant">Hello! I am your Personal Finance Knowledge Assistant. What would you like to know?</div>
        </div>
        <div class="chat-input-container">
            <input type="text" id="chat-input" placeholder="Ask a finance question...">
            <button id="send-btn" title="Send"><div class="send-icon"></div></button>
        </div>
    </div>

    <script>
        const inputField = document.getElementById('chat-input');
        const sendBtn = document.getElementById('send-btn');
        const chatBox = document.getElementById('chat-box');

        async function sendMessage() {
            const userText = inputField.value.trim();
            if (!userText) return;

            chatBox.innerHTML += `<div class="message user">${userText}</div>`;
            inputField.value = '';
            chatBox.scrollTop = chatBox.scrollHeight;

            const thinkingId = 'msg-' + Date.now();
            chatBox.innerHTML += `<div class="message assistant" id="${thinkingId}">Thinking...</div>`;
            chatBox.scrollTop = chatBox.scrollHeight;

            try {
                // MAGIC FIX: Use Colab's native bridge instead of internet fetch!
                const result = await google.colab.kernel.invokeFunction('ask_api', [userText], {});
                const data = result.data['application/json'];

                document.getElementById(thinkingId).innerText = data.answer;
            } catch (error) {
                document.getElementById(thinkingId).innerText = "Error: Internal communication failed.";
                console.error(error);
            }
            chatBox.scrollTop = chatBox.scrollHeight;
        }

        sendBtn.addEventListener('click', sendMessage);
        inputField.addEventListener('keypress', function (e) {
            if (e.key === 'Enter') {
                sendMessage();
            }
        });
    </script>
</body>
</html>
"""

display(HTML(chat_ui_html))

# 📚 Final Explanations & Theory

### Core Concepts Explained Simply
*   **Application:** The front-facing program the user interacts with.
*   **API (Application Programming Interface):** A set of rules that lets different software programs talk to each other.
*   **Ollama:** A local server software that runs LLMs on your own hardware.
*   **Code Llama:** An open-source LLM created by Meta.
*   **Prompt:** The instruction and context text sent to the LLM.
*   **Knowledge Base:** Our trusted folder of text documents containing financial rules.
*   **Document:** A single text file.
*   **Chunking:** Breaking a large document into small, overlapping pieces (chunks) so they fit inside the LLM's limited memory.
*   **Embedding:** Converting text into a mathematical vector (a list of numbers).
*   **Vector:** A coordinate in high-dimensional mathematical space.
*   **FAISS:** Facebook AI Similarity Search. A fast database for storing and searching vectors.
*   **Vector Similarity:** Using math (like Cosine Similarity) to find vectors that point in the same direction, meaning their texts have similar meanings.
*   **Retrieval:** The act of fetching the most mathematically similar chunks to the user's question.
*   **Context:** The retrieved text chunks provided to the LLM.
*   **RAG (Retrieval-Augmented Generation):** Giving the LLM an "open book" (context) to read from before it generates an answer.
*   **LLM (Large Language Model):** An AI trained to understand and generate human language.
*   **Next-token prediction:** The mechanism of an LLM: looking at previous words and calculating the highest probability for the very next word, one word at a time.
*   **Service:** A specific, independent piece of backend software.
*   **API-based communication:** Services talking via HTTP (Web) requests, usually sending JSON data.
*   **Orchestration:** A central API that manages multiple other APIs to fulfill a complex task.
*   **Docker & Container:** Packaging the app and all its dependencies into a virtual box (container) so it runs identically on any computer.

---

### Expected Demonstration Flow
When you type a question in the final demo:
1. **Question:** User asks: *"What is an emergency fund?"*
2. **Query Embedding:** Sentence Transformers converts the string into a 384-dimension vector.
3. **FAISS Search:** FAISS compares the query vector against all chunks in the DB.
4. **Top 3 Chunks:** FAISS returns the closest text chunks.
5. **Context:** The strings are combined.
6. **RAG Prompt:** Application wraps context and question together.
7. **Ollama API:** Application sends JSON to `localhost:11434`.
8. **Code Llama:** Reads the prompt, performs next-token prediction.
9. **Generated Response:** The final answer is sent back to the user.

---

### 5 Demo Questions for the Instructor
1. **Rule check:** *"What is the 50/30/20 budgeting rule?"*
2. **Comparison:** *"What is the difference between simple and compound interest?"*
3. **Definition:** *"What is a credit score and why is it important?"*
4. **Advisory check:** *"How much money should be kept as an emergency fund?"*
5. **Hallucination test:** *"How do I invest in real estate?"* *(Expected: System refuses to answer because it's not in the KB).*

---

### 15 Viva Questions & Short Answers

1. **Why use Ollama?** It allows us to easily run and manage open-source LLMs locally without needing paid cloud APIs.
2. **Why use Code Llama?** It is an excellent Llama 2-based model (required by the assignment) that fits well on local hardware and follows logic/instructions cleanly.
3. **Why use a 7B model?** 7 Billion parameters is the "sweet spot" for running on consumer hardware (like the T4 GPU) while maintaining good reasoning capabilities.
4. **Why use the T4 GPU?** LLMs require massive matrix multiplications. GPUs do this parallel math significantly faster than CPUs.
5. **Why use embeddings?** Computers cannot compare "meaning" between raw text strings, but they can calculate the distance between mathematical vectors.
6. **Why use MiniLM?** `all-MiniLM-L6-v2` is a very lightweight, fast embedding model that works perfectly for small-scale educational applications.
7. **Why FAISS?** Normal databases search exact keywords. FAISS is designed specifically to store and rapidly search high-dimensional vectors for similarity.
8. **Why do we chunk text?** LLMs have a maximum limit of text they can process at once (Context Window). Chunking ensures we only send the most relevant pieces.
9. **What is Cosine Similarity?** A mathematical formula that measures the angle between two vectors. A smaller angle means the texts are semantically more similar.
10. **What is RAG?** Retrieval-Augmented Generation. It means retrieving factual text from a database and giving it to the LLM to generate a safer, factual answer.
11. **Why retrieve top-K chunks?** Retrieving just the top 3 (K=3) ensures the LLM gets enough context without overflowing its memory limit or adding irrelevant noise.
12. **What is hallucination?** When an LLM confidently generates false, invented, or nonsensical information because it lacks correct data in its original training.
13. **What is an API?** A set of rules and addresses (endpoints) that allow two different software programs to talk to each other over the web.
14. **What is orchestration?** A design pattern where a central service coordinates requests between multiple microservices (like RAG and LLM services) to complete a workflow.
15. **What happens if the answer is not in the knowledge base?** Our strict RAG prompt instructs the LLM to admit it doesn't know, preventing it from hallucinating financial advice.